# Sample Superstore — Sales & Profitability Analysis

**End-to-end data analytics project**: data cleaning, exploratory data analysis, SQL-style aggregation, and business recommendations for a U.S. retail superstore's order-level sales extract.


## 1. Project Overview

This project analyzes 9,994 order line items from a U.S. retail "superstore" dataset (the well-known Kaggle *Sample Superstore* dataset), covering sales, profit, discount, and quantity across product categories, customer segments, shipping modes, and U.S. geography.

**Scope note:** the source extract contains no order date, customer ID, seller, or payment fields. Deliverables below are scoped to what the data actually supports — category/geography/segment/shipping analysis and profitability — rather than fabricating time-series or customer-retention metrics the data cannot back up.


## 2. Business Problem

A retail analytics team wants to understand **where the business makes money and where it loses money**, specifically:

- Which product categories and sub-categories drive revenue vs. drag down profit?
- Which regions/states are the strongest and weakest performers?
- Does discounting help or hurt profitability?
- Which customer segments and shipping modes are most valuable?

The goal is a set of data-backed recommendations a merchandising or ops team could act on.


## 3. Dataset Overview

| Property | Value |
|---|---|
| Source file | `SampleSuperstore.csv` |
| Rows | 9,994 (raw) |
| Columns | 13 (raw) |
| Grain | One row per order line item |
| Time range | Not available (no date column) |
| Geography | United States only, 49 states, 531 cities |

**Columns:** Ship Mode, Segment, Country, City, State, Postal Code, Region, Category, Sub-Category, Sales, Quantity, Discount, Profit.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

raw = pd.read_csv("../Data/Raw/superstore_raw.csv")
print(raw.shape)
raw.head()

## 4. Data Quality Assessment

Checking for missing values, duplicates, invalid ranges, and data types before cleaning.


In [ ]:
print("Nulls per column:")
print(raw.isnull().sum())
print("\nExact duplicate rows:", raw.duplicated().sum())
print("\nData types:")
print(raw.dtypes)
print("\nDiscount range:", raw['Discount'].min(), "-", raw['Discount'].max())
print("Quantity range:", raw['Quantity'].min(), "-", raw['Quantity'].max())

**Findings:**
- No missing values in any column.
- **17 exact duplicate rows** — removed during cleaning.
- `Postal Code` is stored as an integer, which strips leading zeros for New England ZIPs (e.g. `02116` becomes `2116`) — needs to be cast to a zero-padded string.
- `Discount` (0–0.8) and `Quantity` (1–14) are within valid business ranges — no invalid values found.
- `Profit` legitimately ranges from **-$6,599.98 to $8,399.98** — large swings are real (bulk discounted furniture orders), not data errors.


## 5. Data Cleaning

Full cleaning logic lives in `Data/clean_data.py`. Summary of steps:

1. Standardized column names to `snake_case`.
2. Dropped 17 duplicate rows.
3. Cast `postal_code` to a zero-padded string (preserves leading zeros).
4. Rounded monetary fields to 2 decimals.
5. Trimmed whitespace on all text fields.
6. Validated `discount` ∈ [0,1] and `quantity` > 0 (none invalid).
7. Flagged (not removed) IQR outliers in `sales`/`profit` — they are real high-value/loss transactions.
8. Engineered `profit_margin`, `unit_price`, and `is_loss_making` columns.
9. Added a surrogate `record_id` primary key (source has none).


In [ ]:
clean = pd.read_csv("../Data/Cleaned/superstore_clean.csv")
print("Rows after cleaning:", len(clean))
clean.head()

## 6. Feature Engineering

`profit_margin = profit / sales`, `unit_price = sales / quantity`, and a boolean `is_loss_making` flag were added to support the profitability analysis below.

In [ ]:
clean[["sales","profit","profit_margin","unit_price"]].describe()

## 7. Database Design & SQL Analysis

A normalized schema (`SQL/create_tables.sql`) was designed around the dimensions the data actually has: `dim_location`, `dim_product`, `dim_ship_mode`, `dim_segment`, and a `fact_sales` fact table. Load logic is in `SQL/import_data.sql`; the full analysis query set is in `SQL/analysis_queries.sql`. Equivalent aggregations are reproduced here in pandas for inline exploration.


In [ ]:
kpis = {
    "Total Revenue": clean["sales"].sum(),
    "Total Profit": clean["profit"].sum(),
    "Total Orders": len(clean),
    "Units Sold": clean["quantity"].sum(),
    "Avg Order Value": clean["sales"].mean(),
    "Overall Margin %": clean["profit"].sum() / clean["sales"].sum() * 100,
}
for k, v in kpis.items():
    print(f"{k:20s}: {v:,.2f}")

## 8. Exploratory Data Analysis

### 8.1 Descriptive Statistics

In [ ]:
clean[["sales","quantity","discount","profit"]].describe()

### 8.2 Correlation Analysis

In [ ]:
corr = clean[["sales","quantity","discount","profit","profit_margin"]].corr()
plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

**Findings:** `sales` and `profit` are moderately positively correlated (r≈0.48) — bigger orders tend to be more profitable in absolute terms. `discount` is strongly *negatively* correlated with `profit_margin` (r≈-0.86) — heavier discounting reliably erodes margin. `discount` has almost no correlation with `sales` volume, suggesting discounts here are not effectively driving incremental demand.


### 8.3 Revenue & Profit by Category

In [ ]:
cat = clean.groupby("category").agg(revenue=("sales","sum"), profit=("profit","sum")).sort_values("revenue", ascending=False)
cat["margin_pct"] = (cat["profit"]/cat["revenue"]*100).round(1)
display(cat)

fig, ax = plt.subplots(1,2, figsize=(12,5))
sns.barplot(x=cat.index, y=cat["revenue"], hue=cat.index, palette="Blues_d", legend=False, ax=ax[0])
ax[0].set_title("Revenue by Category")
sns.barplot(x=cat.index, y=cat["profit"], hue=cat.index, palette="Greens_d", legend=False, ax=ax[1])
ax[1].set_title("Profit by Category")
plt.tight_layout()
plt.show()

**Findings:** Technology and Office Supplies carry the business's profit; Furniture generates strong revenue but a much thinner margin, dragged down by the Tables and Bookcases sub-categories (see below).


### 8.4 Loss-Making Sub-Categories

In [ ]:
sub_profit = clean.groupby("sub_category")["profit"].sum().sort_values()
print("Bottom 5 sub-categories by total profit:")
display(sub_profit.head(5))

plt.figure(figsize=(8,6))
colors = ["#d62728" if v < 0 else "#2ca02c" for v in sub_profit.values]
plt.barh(sub_profit.index, sub_profit.values, color=colors)
plt.title("Profit by Sub-Category (red = net loss)")
plt.tight_layout()
plt.show()

### 8.5 Discount vs. Profitability

In [ ]:
bins = [-0.01, 0, 0.2, 0.4, 1.0]
labels = ["0%","1-20%","21-40%","41%+"]
clean["discount_band"] = pd.cut(clean["discount"], bins=bins, labels=labels)
band = clean.groupby("discount_band", observed=True).agg(orders=("record_id","count"), avg_profit=("profit","mean"))
display(band)

plt.figure(figsize=(7,5))
sns.barplot(x=band.index, y=band["avg_profit"], hue=band.index, palette="rocket", legend=False)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Average Profit per Order by Discount Band")
plt.ylabel("Avg Profit ($)")
plt.tight_layout()
plt.show()

**Findings:** average profit per order turns **negative once discounts exceed ~20%**. Discounts above 40% are, on average, actively unprofitable per order.


### 8.6 Geographic Performance

In [ ]:
region = clean.groupby("region")["sales"].sum().sort_values(ascending=False)
top_states = clean.groupby("state")["sales"].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(1,2, figsize=(13,5))
ax[0].pie(region.values, labels=region.index, autopct="%1.1f%%", colors=sns.color_palette("Set2"))
ax[0].set_title("Revenue Share by Region")
sns.barplot(x=top_states.values, y=top_states.index, hue=top_states.index, palette="crest", legend=False, ax=ax[1])
ax[1].set_title("Top 10 States by Revenue")
plt.tight_layout()
plt.show()

### 8.7 Segment & Shipping Analysis

In [ ]:
seg = clean.groupby("segment").agg(revenue=("sales","sum"), profit=("profit","sum"), orders=("record_id","count"))
seg["margin_pct"] = (seg["profit"]/seg["revenue"]*100).round(1)
display(seg)

ship = clean["ship_mode"].value_counts()
plt.figure(figsize=(7,4))
sns.barplot(x=ship.values, y=ship.index, hue=ship.index, palette="flare", legend=False)
plt.title("Order Count by Ship Mode")
plt.tight_layout()
plt.show()

## 9. Business Insights — Top 10 Findings

1. Overall business margin is **12.5%** on **$2.30M** revenue across 9,977 cleaned order lines.
2. **Technology** is the strongest category by profit margin; **Furniture** is the weakest despite solid revenue.
3. **Tables and Bookcases** are net loss-makers overall — the single biggest profitability drag in the catalog.
4. Discounting beyond **~20%** flips average per-order profit negative; beyond **40%** losses are the norm, not the exception.
5. Discount level has **almost no correlation with sales volume** — heavy discounts here are not buying meaningful extra demand.
6. The **Consumer segment** generates the most revenue but a slightly *lower* margin than Home Office.
7. **West and East regions** outperform South and Central in total revenue.
8. Revenue is geographically concentrated: the **top 10 states** account for a disproportionate share of total sales.
9. **Standard Class** is by far the dominant shipping mode by order count, well ahead of Same Day.
10. ~19% of all order lines are **loss-making** (`profit < 0`), concentrated in high-discount Furniture orders.

## 10. Recommendations

1. Cap discounts on Furniture (especially Tables, Bookcases) at ~20% pending a pricing review.
2. Re-price or discontinue chronically loss-making Tables/Bookcases SKUs, or renegotiate supplier cost.
3. Shift promotional budget toward Technology and Office Supplies, where margin supports discounting.
4. Investigate why discounts aren't lifting volume — consider testing bundling instead of blanket price cuts.
5. Double down on marketing spend in top-performing states/regions (West, East) where CAC is likely lower per dollar of revenue.
6. Run a targeted regional campaign for South/Central to close the revenue gap.
7. Evaluate whether Home Office's higher margin (vs. Consumer) reflects an underserved, more profitable segment worth growing.
8. Set a per-order discount approval threshold (e.g. >30% requires manager sign-off) to stop silent margin leakage.
9. Track shipping-mode profitability going forward once cost data is available — Same Day likely carries higher fulfillment cost per order.
10. Because this extract has no order date, prioritize collecting timestamped transaction data next quarter to enable trend, seasonality, and retention analysis.

## 11. Conclusion

The Superstore dataset shows a fundamentally profitable business (12.5% margin) with a clear, fixable problem: **aggressive discounting on already-thin-margin Furniture sub-categories is destroying value**. Category mix and geography offer more upside than further price cuts. The recommendations above are ranked to prioritize the highest-leverage, lowest-effort actions first — capping Furniture discounts and re-pricing the worst-performing SKUs.
